# Notebook 04: Real-Time Fraud Detection

## Why This Matters

Fraud is a multi-hundred-billion-dollar problem: global payment fraud losses exceeded $32B in 2022. Every major payment processor (Stripe, PayPal, Visa, Mastercard) runs sophisticated real-time ML fraud detection. The design constraints are uniquely challenging: score a transaction in <100ms (synchronous with the payment flow), handle extreme class imbalance (fraud rates 0.1%-1%), and fight an adversary that actively adapts to your model. Understanding this system is essential for fintech, banking, and platform trust roles.

In [ ]:
%pip install -q scikit-learn xgboost imbalanced-learn matplotlib pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, roc_auc_score,
                              average_precision_score, classification_report)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)
print("Ready.")

## 1. Why Fraud Detection Is Hard

**Class imbalance**: Fraud rates of 0.1%-1% mean a model predicting 'not fraud' achieves 99%+ accuracy. Accuracy is useless. Use PR-AUC.

**Adversarial nature**: Fraudsters reverse-engineer detection patterns, rotate accounts, use stolen legitimate cards, and adapt within days. Your model is in an arms race.

**Latency constraint**: Transaction decisions must complete in <100ms end-to-end. No batch processing - you need a real-time feature pipeline and a fast model (GBM, not deep neural net).

**Delayed ground truth**: Chargebacks arrive hours to days after the transaction. Your training labels arrive late.

In [ ]:
def generate_fraud_dataset(n_samples=100000, fraud_rate=0.008):
    n_fraud = int(n_samples * fraud_rate)
    n_legit = n_samples - n_fraud
    legit = pd.DataFrame({
        "amount": np.random.lognormal(4, 1.5, n_legit),
        "hour": np.random.choice(range(24), n_legit),
        "velocity_1h": np.random.poisson(2, n_legit),
        "velocity_24h": np.random.poisson(8, n_legit),
        "is_foreign": np.random.binomial(1, 0.05, n_legit),
        "is_new_merchant": np.random.binomial(1, 0.10, n_legit),
        "time_since_last_txn_min": np.random.exponential(180, n_legit),
        "user_avg_amount": np.random.lognormal(4, 0.5, n_legit),
        "label": 0
    })
    fraud = pd.DataFrame({
        "amount": np.random.lognormal(5, 1.0, n_fraud),
        "hour": np.random.choice(range(24), n_fraud),
        "velocity_1h": np.random.poisson(8, n_fraud),
        "velocity_24h": np.random.poisson(20, n_fraud),
        "is_foreign": np.random.binomial(1, 0.40, n_fraud),
        "is_new_merchant": np.random.binomial(1, 0.60, n_fraud),
        "time_since_last_txn_min": np.random.exponential(20, n_fraud),
        "user_avg_amount": np.random.lognormal(3.5, 0.5, n_fraud),
        "label": 1
    })
    return pd.concat([legit, fraud]).sample(frac=1, random_state=42)

df = generate_fraud_dataset()
print(f"Dataset: {df.shape}, Fraud rate: {df.label.mean():.3%}")

## 2. Handling Class Imbalance

Three main approaches:
1. **class_weight='balanced'**: upweights minority class in the loss. Simple, try first.
2. **SMOTE**: generate synthetic fraud examples by interpolating. More data but synthetic.
3. **Threshold tuning**: train normally, move decision threshold to optimize your cost function.

**Cost-sensitive thinking**: In fraud, a false negative (missed fraud) might cost $500. A false positive (blocking legitimate transaction) might cost $0.50 in friction. Your threshold should reflect this asymmetry - never set at 0.5.

In [ ]:
features = ["amount", "hour", "velocity_1h", "velocity_24h", "is_foreign", "is_new_merchant", "time_since_last_txn_min", "user_avg_amount"]
X = df[features].values; y = df.label.values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Add engineered feature: amount / user_avg_amount ratio
def add_features(X):
    ratio = X[:, 0] / (X[:, -1] + 1e-6)
    return np.column_stack([X, ratio, np.log1p(X[:, 0])])

X_train_eng = add_features(X_train); X_test_eng = add_features(X_test)
scale_pos = (y_train==0).sum() / (y_train==1).sum()

model_xgb = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos, eval_metric="aucpr", random_state=42, verbosity=0)
model_xgb.fit(X_train_eng, y_train, eval_set=[(X_test_eng, y_test)], verbose=False)
xgb_probs = model_xgb.predict_proba(X_test_eng)[:, 1]

print(f"=== XGBoost (scale_pos_weight={scale_pos:.1f}) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, xgb_probs):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, xgb_probs):.4f}")
print()
print("PR-AUC is the key metric for fraud - not ROC-AUC, not accuracy.")

## 3. Threshold Optimization

The default threshold of 0.5 is almost always wrong for imbalanced problems. The optimal threshold depends on your cost function:

- **Maximize F1**: balance precision and recall equally
- **Maximize precision @ 80% recall**: don't miss 20% of fraud, minimize false positives
- **Minimize total cost**: use a cost matrix with FN cost and FP cost

**Interview tip**: Always mention threshold optimization when discussing fraud systems. Many candidates skip this.

In [ ]:
def optimize_threshold(y_true, y_probs, fn_cost=500, fp_cost=0.5):
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    results = []
    for i, thresh in enumerate(thresholds):
        preds = (y_probs >= thresh).astype(int)
        tp = ((preds==1)&(y_true==1)).sum(); fp = ((preds==1)&(y_true==0)).sum()
        fn = ((preds==0)&(y_true==1)).sum()
        total_cost = fn * fn_cost + fp * fp_cost
        f1 = 2*precision[i]*recall[i]/(precision[i]+recall[i]+1e-8)
        results.append({"threshold": thresh, "precision": precision[i], "recall": recall[i], "f1": f1, "total_cost": total_cost})
    df_r = pd.DataFrame(results)
    best_cost_idx = df_r.total_cost.idxmin(); best_f1_idx = df_r.f1.idxmax()
    print("=== Threshold Optimization ===")
    default_row = df_r[df_r.threshold <= 0.5].iloc[-1]
    print(f"Default (0.5):    Prec={default_row.precision:.3f}, Recall={default_row.recall:.3f}, Cost=${default_row.total_cost:,.0f}")
    row = df_r.loc[best_f1_idx]; print(f"Best F1 (t={row.threshold:.3f}): Prec={row.precision:.3f}, Recall={row.recall:.3f}, Cost=${row.total_cost:,.0f}")
    row = df_r.loc[best_cost_idx]; print(f"Cost-opt (t={row.threshold:.3f}):Prec={row.precision:.3f}, Recall={row.recall:.3f}, Cost=${row.total_cost:,.0f}")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(recall, precision, "b-", lw=2)
    ax1.scatter([df_r.loc[best_cost_idx, "recall"]], [df_r.loc[best_cost_idx, "precision"]], s=100, color="red", label="Cost-optimal")
    ax1.set_xlabel("Recall"); ax1.set_ylabel("Precision"); ax1.set_title("Precision-Recall Curve"); ax1.legend()
    ax2.plot(df_r.threshold, df_r.total_cost/1000, "r-")
    ax2.axvline(df_r.loc[best_cost_idx, "threshold"], color="gray", linestyle="--", label="Optimal")
    ax2.set_xlabel("Threshold"); ax2.set_ylabel("Cost ($K)"); ax2.set_title("Cost vs Threshold"); ax2.legend()
    plt.tight_layout(); plt.savefig("/tmp/fraud_threshold.png", dpi=80); plt.show()

optimize_threshold(y_test, xgb_probs)

## 4. Concept Drift in Fraud

Fraud patterns are non-stationary. Fraudsters adapt to your model within weeks. This is **concept drift**: P(fraud | features) changes over time.

**Detection signals**:
- Model score distribution shifts (mean score decreases while chargeback rate holds)
- Feature importance shifts (new fraud vectors emerging)
- Precision drops while recall holds

**Response**: Trigger retraining when PSI on model scores exceeds 0.25, or when precision drops >5% on a holdout window. Maintain a fast retraining pipeline (daily or weekly, not monthly).

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Class imbalance | Accuracy is useless; use PR-AUC |
| scale_pos_weight | XGBoost parameter for imbalance; = n_negative / n_positive |
| Threshold tuning | Optimal threshold != 0.5; use cost matrix |
| Velocity features | Transactions per hour/day per user - key fraud signal |
| Concept drift | Fraudsters adapt; retrain on rolling windows, monitor PSI |

### Common Pitfalls
- Reporting accuracy instead of PR-AUC on imbalanced fraud data
- Applying SMOTE before train/test split (leakage)
- Not monitoring for concept drift after deployment
- Setting threshold at 0.5 without considering business costs
